# Multi-state posts: LLM vs Stanford NER (Colab Pro)

Cells 1-7 run the LLM test (prompt v2), cell 8 Stanford NER, cell 9 scores both against labels.csv, cell 10 applies the chosen method to every Meta and X post that names a US state. Documented in docs/Geolocation_Extraction.docx, section 6. Choose **Runtime → Change runtime type → A100 GPU** before running.

## Cell 1: GPU check and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## Cell 2: install Ollama and start it in the background

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time, requests
# OLLAMA_NUM_PARALLEL lets cell 10 send 4 posts at once; it does not change answers.
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_NUM_PARALLEL": "4"})
for _ in range(60):
    try:
        print("Ollama", requests.get("http://localhost:11434/api/version").json()["version"])
        break
    except requests.ConnectionError:
        time.sleep(1)

## Cell 3: download the model (about 17 GB; a few minutes)

In [ ]:
MODEL = "gemma3:27b"         # the model chosen in testing; "qwen3:32b" also fits an A100
!ollama pull {MODEL}

## Cell 4: connect to the database (same as notebooks/colab_x_corpus.ipynb)

In [ ]:
!pip install -q "cloud-sql-python-connector[pg8000]" sqlalchemy pandas
from google.colab import auth
auth.authenticate_user()     # sign in with your @g.harvard.edu account

import getpass
import pandas as pd
import sqlalchemy
from google.cloud.sql.connector import Connector

INSTANCE = "harvard-capstone-499102:us-central1:h5n1-pg"
DB_PASSWORD = getpass.getpass("h5n1_app password: ")
connector = Connector()

def _getconn():
    return connector.connect(INSTANCE, "pg8000", user="h5n1_app",
                             password=DB_PASSWORD, db="h5n1")

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=_getconn,
                                  pool_pre_ping=True)
print(pd.read_sql("SELECT count(*) FROM meta_post_analysis WHERE geo_tier = 'T5_multi'",
                  engine).iloc[0, 0], "T5 posts")

## Cell 5: the prompt: reference list, rules and answer format

In [ ]:
# Copied from h5n1/sources/meta_location.py and scripts/llm_state_pilot.py.
# Keep them identical, or Colab and local results stop being comparable.
STATE_NAME = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "florida": "FL", "georgia": "GA", "hawaii": "HI", "idaho": "ID",
    "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN",
    "mississippi": "MS", "missouri": "MO", "montana": "MT", "nebraska": "NE",
    "nevada": "NV", "new hampshire": "NH", "new jersey": "NJ",
    "new mexico": "NM", "new york": "NY", "north carolina": "NC",
    "north dakota": "ND", "ohio": "OH", "oklahoma": "OK", "oregon": "OR",
    "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA",
    "west virginia": "WV", "wisconsin": "WI", "wyoming": "WY",
    "district of columbia": "DC",
}
SPANISH_STATE = {
    "nueva york": "NY", "carolina del norte": "NC", "carolina del sur": "SC",
    "dakota del norte": "ND", "dakota del sur": "SD", "nuevo mexico": "NM",
    "nuevo méxico": "NM", "luisiana": "LA", "pensilvania": "PA",
    "virginia occidental": "WV", "virginia del oeste": "WV",
    "nueva jersey": "NJ", "nuevo hampshire": "NH", "misisipi": "MS",
    "misisipí": "MS", "misuri": "MO", "hawái": "HI",
    "nueva inglaterra": None,
    "tejas": "TX", "georgia": "GA",
}
CODES = sorted(set(STATE_NAME.values()))
_es = {}
for name, code in SPANISH_STATE.items():
    if code:
        _es.setdefault(code, []).append(name)
REFERENCE = "\n".join(
    f"{code} = {en.title()}" + (f" (es: {', '.join(n.title() for n in _es[code])})"
                                if code in _es else "")
    for en, code in sorted(STATE_NAME.items(), key=lambda kv: kv[1]))

SYSTEM = f"""You read social-media posts about avian influenza (bird flu). Posts are in English or Spanish.

Task: find the US states where the post reports a bird-flu EVENT. An event is a detection in birds or animals, a human case, an outbreak, a cull, a quarantine, or a state's own response such as an emergency declaration.

Rules:
- Use only codes from the reference list below.
- A post can report events in several states. List every one.
- Do NOT count a state that is only part of another place's name: Lake Michigan, Kansas City, Washington County (Oregon), Georgia del Sur / South Georgia, Baja California, Sierra Nevada, La Florida (Chile).
- Do NOT count a state name that is part of the name of a species, breed, product, company or publication: California condor, Carolina wren, Mississippi kite, Rhode Island Red (a chicken breed), Jersey cows, Kentucky Fried Chicken, The New York Times, The Washington Post. Place the event where it happened: a California condor that died in Arizona is an Arizona event.
- Do NOT count "Washington" when it means the US federal government.
- Do NOT count a state named only for contrast, history or comparison ("unlike Iowa, we have no cases").
- Places outside the US are not states. If the event is outside the US, return no states.
- If the post is about the US as a whole with no specific state, return no states.

states: every US state with an event, or [] if none.
main_state: the ONE state the post is mainly about -- where its main event happened. If several states are covered equally, the one the post names first. It must be one of the codes in states. Use "" when states is empty.
us_event: true if the post reports a specific bird-flu event inside the US, else false.
evidence: the shortest phrase from the post (at most 15 words, original language) that supports your answer, or "" if none.

Reference list of US states:
{REFERENCE}"""

SCHEMA = {
    "type": "object",
    "properties": {
        "states": {"type": "array", "items": {"type": "string", "enum": CODES}},
        "main_state": {"type": "string", "enum": CODES + [""]},
        "us_event": {"type": "boolean"},
        "evidence": {"type": "string"},
    },
    "required": ["states", "main_state", "us_event", "evidence"],
}

## Cell 6: draw the sample (same 200 posts every time) and save it to Drive

In [ ]:
import json
from pathlib import Path
from sqlalchemy import text

OUT = Path("/content/drive/MyDrive/h5n1_llm_pilot")
OUT.mkdir(parents=True, exist_ok=True)
SEED, N_PER_LANG, TIER = "pilot1", 100, "T5_multi"

sample_path = OUT / "sample.csv"
if sample_path.exists():
    sample = pd.read_csv(sample_path, dtype=str, keep_default_na=False)
else:
    sample = pd.read_sql(text("""
        SELECT post_id, lang, geo_tier, geo_matched, post_text FROM (
            SELECT *, row_number() OVER (PARTITION BY lang
                                         ORDER BY md5(post_id::text || :seed)) AS r
            FROM meta_post_analysis WHERE geo_tier = :tier) s
        WHERE r <= :n ORDER BY lang, r"""),
        engine, params={"tier": TIER, "n": N_PER_LANG, "seed": SEED})
    sample.insert(0, "row", range(1, len(sample) + 1))   # join key for labels
    sample["post_id"] = sample.post_id.astype(str)
    sample["human_states"] = ""      # you fill these two, in a copy named labels.csv
    sample["human_event"] = ""
    sample.to_csv(sample_path, index=False)
if "human_main" not in sample.columns:         # added with prompt v2
    sample.insert(sample.columns.get_loc("human_states") + 1, "human_main", "")
    sample.to_csv(sample_path, index=False)
print(len(sample), "posts", sample.lang.value_counts().to_dict())

## Cell 7: run the model; saves every 10 posts and resumes where it stopped

In [ ]:
PROMPT_VERSION = "v2"     # v2 adds main_state; bump it whenever SYSTEM changes
MAX_CHARS = 2000
USE_HINT = True
THINK = False if MODEL.startswith(("qwen3", "gpt-oss", "deepseek")) else None

def ask(post, hint):
    user = post[:MAX_CHARS]
    if hint:
        user += f"\n\n(A keyword search found these state names in the text: {hint}. They may be wrong.)"
    body = {"model": MODEL, "stream": False, "format": SCHEMA,
            "options": {"temperature": 0, "num_ctx": 4096},
            "messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": user}]}
    if THINK is not None:
        body["think"] = THINK
    r = requests.post("http://localhost:11434/api/chat", json=body, timeout=600)
    r.raise_for_status()
    return json.loads(r.json()["message"]["content"])

def with_text(df):
    """Put the full post beside each answer, so the results file reads on its own."""
    df = df.drop(columns="post_text", errors="ignore").merge(
        sample[["row", "post_text"]].astype(str), on="row", how="left")
    cols = [c for c in df.columns if c != "post_text"]
    cols.insert(cols.index("geo_matched") + 1, "post_text")
    return df[cols]

res_path = OUT / f"{MODEL.replace(':', '_').replace('/', '_')}_{PROMPT_VERSION}.csv"
done = (pd.read_csv(res_path, dtype=str, keep_default_na=False)
        if res_path.exists() else pd.DataFrame())
if len(done):
    done = done[done.error == ""]   # posts that failed last time are tried again
    if "post_text" not in done.columns:   # files made before the text column was added
        done = with_text(done)
        done.to_csv(res_path, index=False)
todo = sample[~sample["row"].astype(str).isin(set(done.get("row", [])))]
print(f"{len(done)} done, {len(todo)} to go")

rows = []
for i, p in enumerate(todo.itertuples(), 1):
    t0 = time.time()
    try:
        a, err = ask(p.post_text, p.geo_matched if USE_HINT else None), ""
    except Exception as e:      # one bad reply should not end the run
        a, err = {"states": [], "main_state": "", "us_event": None, "evidence": ""}, str(e)[:200]
    states, main = set(a.get("states", [])), a.get("main_state", "")
    if main:
        states.add(main)        # the main state is always one of the states
    rows.append({"row": p.row, "post_id": p.post_id, "lang": p.lang,
                 "geo_matched": p.geo_matched,
                 "llm_states": ",".join(sorted(states)), "llm_main": main,
                 "llm_event": a.get("us_event"), "evidence": a.get("evidence", ""),
                 "seconds": round(time.time() - t0, 1), "error": err})
    if i % 10 == 0 or i == len(todo):
        done = with_text(pd.concat([done, pd.DataFrame(rows).astype(str)], ignore_index=True))
        done.to_csv(res_path, index=False)
        rows = []
        print(f"  {len(done)}/{len(sample)} saved  (last {done.seconds.iloc[-1]}s per post)")

print(f"\n{res_path.name}: errors {(done.error != '').sum()}, "
      f"median {pd.to_numeric(done.seconds).median()}s per post")

## Cell 8: Stanford NER (Stanza) on the same sample, for comparison

In [ ]:
!pip install -q stanza
import re, stanza
from collections import Counter

# French names are only needed for X's French posts in cell 10.
FRENCH_STATE = {
    "californie": "CA", "floride": "FL", "g\u00e9orgie": "GA", "louisiane": "LA",
    "pennsylvanie": "PA", "virginie": "VA", "virginie-occidentale": "WV",
    "virginie occidentale": "WV", "caroline du nord": "NC", "caroline du sud": "SC",
    "dakota du nord": "ND", "dakota du sud": "SD", "nouveau-mexique": "NM",
    "nouveau mexique": "NM", "nouvelle-orl\u00e9ans": "LA", "hawa\u00ef": "HI",
    "new hampshire": "NH",
}
ALL_NAMES = {**STATE_NAME, **{k: v for k, v in SPANISH_STATE.items() if v}, **FRENCH_STATE}
PLACE_TYPES = {"GPE", "LOC"}

# Wrappers around a state name that still mean the state: "the State of Texas",
# "el estado de Texas", "Washington state", "New York State".
_PREFIX = re.compile(r"^(the\s+)?(state\s+of|estado\s+de|el\s+estado\s+de|[ée]tat\s+de)\s+", re.I)
_SUFFIX = re.compile(r"\s+state$", re.I)


def to_state(entity: str) -> str | None:
    """'Texas' / 'Nueva York' / 'Washington state' -> code. Anything else -> None."""
    t = _SUFFIX.sub("", _PREFIX.sub("", entity.strip(" .,;:()\"'"))).lower()
    if t in ("washington d.c", "washington, d.c", "washington dc", "d.c", "dc"):
        return "DC"
    return ALL_NAMES.get(t)


def label(nlp, text: str) -> tuple[list[str], str, str]:
    """(states, main state, the place entities found) for one post."""
    doc = nlp(text)
    places = [e.text for e in doc.ents if e.type in PLACE_TYPES]
    hits = [s for s in map(to_state, places) if s]
    if not hits:
        return [], "", "; ".join(places[:15])
    counts = Counter(hits)
    top = max(counts.values())
    main = next(s for s in hits if counts[s] == top)   # most mentioned; ties -> first
    return sorted(counts), main, "; ".join(places[:15])

pipes = {}
for lang in ("en", "es"):
    stanza.download(lang, processors="tokenize,ner", verbose=False)
    pipes[lang] = stanza.Pipeline(lang, processors="tokenize,ner", verbose=False)

ner_rows = []
for p in sample.itertuples():
    t0 = time.time()
    states, main, places = label(pipes[p.lang], p.post_text)
    ner_rows.append({"row": p.row, "post_id": p.post_id, "lang": p.lang,
                     "geo_matched": p.geo_matched, "post_text": p.post_text,
                     "llm_states": ",".join(states), "llm_main": main, "llm_event": "",
                     "evidence": places, "seconds": round(time.time() - t0, 1), "error": ""})
pd.DataFrame(ner_rows).to_csv(OUT / "stanza_ner.csv", index=False)
print("wrote stanza_ner.csv")

## Cell 9: score every method against labels.csv (upload it to the Drive folder first)

In [ ]:
# Same code as scripts/score_state_methods.py.
SKIP = {"sample.csv", "labels.csv"}


def _set(s) -> frozenset:
    return frozenset(x.strip().upper() for x in str(s or "").split(",") if x.strip())


def score(folder: Path, high_confidence: bool = False) -> pd.DataFrame:
    lab = pd.read_csv(folder / "labels.csv", dtype=str, keep_default_na=False)
    lab["row"] = lab["row"].str.strip()
    lab = lab[lab.human_event.str.strip() != ""]
    if high_confidence and "confidence" in lab.columns:
        lab = lab[lab.confidence != "low"]

    methods = {"keyword_gazetteer": lab.assign(llm_states=lab.geo_matched, llm_main="",
                                               llm_event="")}
    for f in sorted(folder.glob("*.csv")):
        if f.name in SKIP:
            continue
        m = pd.read_csv(f, dtype=str, keep_default_na=False)
        if "llm_main" not in m.columns:
            print(f"{f.stem}: no main state (made before prompt v2), skipped")
            continue
        m = (m.drop(columns=["lang", "geo_matched", "post_text"], errors="ignore")
              .rename(columns={"post_id": "res_post_id"}).merge(lab, on="row"))
        # Rows must be the same posts. Skip the check where Excel has mangled post_id.
        if "res_post_id" in m.columns:
            clean = m.post_id.str.fullmatch(r"\d+")
            bad = clean & (m.res_post_id != m.post_id)
            if bad.any():
                print(f"{f.stem}: {bad.sum()} rows are different posts from labels.csv "
                      f"(sample drawn differently?) -- left out")
                m = m[~bad]
        methods[f.stem] = m

    out = []
    for name, m in methods.items():
        for lang, g in [("all", m)] + sorted(m.groupby("lang")):
            truth, pred = g.human_states.map(_set), g.llm_states.map(_set)
            tp = sum(len(t & p) for t, p in zip(truth, pred))
            n_pred, n_true = sum(map(len, pred)), sum(map(len, truth))
            answered = g.llm_event.str.strip() != ""
            ev = ((g.llm_event[answered].str.lower().map({"true": "1", "false": "0"})
                   == g.human_event[answered].str.strip()).mean() if answered.any() else None)
            out.append({
                "method": name, "lang": lang, "n": len(g),
                "main state": (g.llm_main.str.upper().str.strip()
                               == g.human_main.str.upper().str.strip()).mean(),
                "exact list": (truth == pred).mean(),
                "precision": tp / n_pred if n_pred else None,
                "recall": tp / n_true if n_true else None,
                "event": ev,
            })
    return pd.DataFrame(out)


def show(df: pd.DataFrame) -> None:
    pct = lambda v: "   -  " if v is None or pd.isna(v) else f"{v:6.1%}"
    print(f"{'method':22} {'lang':4} {'n':>4}  {'main state':>10} {'exact list':>10} "
          f"{'precision':>9} {'recall':>7} {'event':>7}")
    for r in df.itertuples():
        print(f"{r.method:22} {r.lang:4} {r.n:>4}  {pct(r[4]):>10} {pct(r[5]):>10} "
              f"{pct(r.precision):>9} {pct(r.recall):>7} {pct(r.event):>7}")

show(score(OUT))
print("\nhigh-confidence labels only:")
show(score(OUT, high_confidence=True))

## Cell 10: apply the chosen method to every Meta and X post that names a US state

In [ ]:
# Runs cells 1-5 first (and cell 8 for METHOD = "ner"). Saves to Drive every 200 posts
# and resumes where it stopped. Nothing is written to the database here.
from concurrent.futures import ThreadPoolExecutor

METHOD = "llm"           # "llm" uses MODEL from cell 3 with the cell 5 prompt; "ner" uses cell 8
WORKERS = 4              # posts sent to Ollama at once (cell 2 allows 4)
TIERS = "('T1_county_state','T2_city_state','T3_state','T4_county_unique','T5_multi')"
SOURCES = {"meta": "meta_post_analysis", "x": "x_post_clean"}
TAG = (f"{MODEL.replace(':', '_')}_{PROMPT_VERSION}" if METHOD == "llm" else "stanza_ner")

if METHOD == "ner" and "fr" not in pipes:
    stanza.download("fr", processors="tokenize,ner", verbose=False)
    pipes["fr"] = stanza.Pipeline("fr", processors="tokenize,ner", verbose=False)

def run_one(p):
    try:
        if METHOD == "llm":
            a = ask(p.post_text, p.geo_matched if USE_HINT else None)
            states, main = set(a.get("states", [])), a.get("main_state", "")
            if main:
                states.add(main)
            return {"post_id": p.post_id, "llm_states": ",".join(sorted(states)),
                    "llm_main": main, "llm_event": a.get("us_event"),
                    "evidence": a.get("evidence", ""), "error": ""}
        states, main, places = label(pipes.get(p.lang, pipes["en"]), p.post_text)
        return {"post_id": p.post_id, "llm_states": ",".join(states), "llm_main": main,
                "llm_event": "", "evidence": places, "error": ""}
    except Exception as e:
        return {"post_id": p.post_id, "llm_states": "", "llm_main": "", "llm_event": None,
                "evidence": "", "error": str(e)[:200]}

for corpus, table in SOURCES.items():
    exists = pd.read_sql(f"SELECT to_regclass('public.{table}') IS NOT NULL AS ok", engine).ok[0]
    if not exists:
        print(f"{table} does not exist yet -- build it first (scripts/build_x_clean.py)")
        continue
    posts = pd.read_sql(f"SELECT post_id, lang, geo_tier, geo_matched, post_text "
                        f"FROM {table} WHERE geo_tier IN {TIERS}", engine)
    path = OUT / f"full_{corpus}_{TAG}.csv"
    done = (pd.read_csv(path, dtype=str, keep_default_na=False)
            if path.exists() else pd.DataFrame(columns=["post_id", "error"]))
    done = done[done.error == ""]
    todo = posts[~posts.post_id.astype(str).isin(set(done.post_id))]
    print(f"{corpus}: {len(posts):,} posts name a state; {len(done):,} done, {len(todo):,} to go")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS if METHOD == "llm" else 1) as pool:
        for start in range(0, len(todo), 200):
            chunk = todo.iloc[start:start + 200]
            new = pd.DataFrame(list(pool.map(run_one, chunk.itertuples())))
            new = new.merge(chunk[["post_id", "lang", "geo_tier", "geo_matched"]]
                            .astype({"post_id": str}), on="post_id")
            done = pd.concat([done, new.astype(str)], ignore_index=True)[
                ["post_id", "lang", "geo_tier", "geo_matched", "llm_states", "llm_main",
                 "llm_event", "evidence", "error"]]
            done.to_csv(path, index=False)
            rate = (start + len(chunk)) / (time.time() - t0)
            print(f"  {len(done):,}/{len(posts):,}  ({rate:.1f} posts/s, "
                  f"~{(len(todo) - start - len(chunk)) / rate / 3600:.1f} h left)")
    print(f"wrote {path.name}: {len(done):,} rows, errors {(done.error != '').sum()}")

# Loading into the database is a separate, deliberate step, e.g.
#   pd.read_csv(OUT / f"full_meta_{TAG}.csv", dtype=str).to_sql(
#       "meta_post_state_llm", engine, if_exists="replace", index=False, chunksize=1000)